# 5. Main Regression Analysis: Wealth x Cardiometabolic Burden Interaction

**Research question:** does cardiometabolic burden (diabetes, hypertension, obesity) modify the
association between household wealth and probable depression / probable anxiety among currently
married women in Bangladesh (BDHS 2022)?

**Design:** survey-weighted logistic regression, two outcome models (depression, anxiety) run in
parallel, each a nested Model 1 (main effects) -> Model 2 (+ Wealth x Burden interaction), plus a
Model 3 sensitivity check with a coarser coding of the interaction. n = 4,887 currently married
women for both outcomes -- see `docs/Dictionary.md` and `Figure-2` for the derivation and
attrition.

"Socioeconomic status" here means the wealth index specifically, not a composite of wealth and
education -- that's how the variable is named and coded throughout this pipeline. Education stays
in the model as a covariate/confounder, the same role it plays in the DAG (`Figure-1`), rather than
as a second focal moderator. A parallel education-focal analysis (Education x Burden, Wealth as
covariate) would be a natural extension of this notebook if a future paper wants it, but it isn't
part of the current one.

Python (`statsmodels`) is the working environment here, not the final one. Two real gaps versus R's
`survey` package or Stata's `svy:` prefix are worth flagging up front, since they shape how the rest
of the notebook is structured:

- `statsmodels` has no stratified-multistage variance estimator -- it clusters by PSU fine, but
  doesn't incorporate `Stratum` into the variance at all.
- `statsmodels` itself documents its cluster-robust covariance as not fully supported in combination
  with weights.

So the plan is: cluster-robust-by-PSU as the Python-side working estimate, a stratified cluster
bootstrap (resampling PSUs within stratum, which *does* respect the two-stage design) as a
cross-check, and exact R `survey::svyglm` / Stata `svy:` syntax at the end for the numbers that
actually go in the manuscript. None of this affects the odds ratios themselves, only their standard
errors and p-values.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests
import patsy
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

dep = pd.read_csv('../resources/depression_dataset.csv')
anx = pd.read_csv('../resources/anxiety_dataset.csv')

print('Depression analytic sample:', dep.shape)
print('Anxiety analytic sample:   ', anx.shape)
assert dep.shape[0] == anx.shape[0] == 4887, "Sample size drifted from the expected n = 4,887 -- re-run Notebooks 2-4 before proceeding."


Depression analytic sample: (4887, 27)
Anxiety analytic sample:    (4887, 27)


## 2. Outcome definition

`Depression` and `Anxiety` in the analytic datasets are severity bands (PHQ-9, GAD-7), not binary
case indicators -- see `docs/Dictionary.md`. Survey-weighted logistic regression needs a binary
outcome, so each gets dichotomized at the standard clinical screening threshold:

- **Probable depression** = PHQ-9 >= 10, i.e. `Depression` band in {2, 3, 4} (moderate or worse).
  Kroenke, Spitzer & Williams (2001), *J Gen Intern Med*.
- **Probable anxiety** = GAD-7 >= 10, i.e. `Anxiety` band in {2, 3} (moderate or worse).
  Spitzer, Kroenke, Williams & Lowe (2006), *Arch Intern Med*.

This is a choice, not something the data forces -- a lower cutoff (any symptoms, band >= 1) or
keeping the full severity band as an ordinal outcome would both be defensible too. Section 20 below
re-runs the depression model at a higher cutoff (PHQ-9 >= 15) as a check on how much this matters;
both cutoffs and their citations belong in the Methods writeup regardless.

In [2]:
dep['Depression_binary'] = (dep['Depression'] >= 2).astype(int)
anx['Anxiety_binary'] = (anx['Anxiety'] >= 2).astype(int)

for name, df, col in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    n_events = df[col].sum()
    unweighted_prev = df[col].mean()
    weighted_prev = np.average(df[col], weights=df['Sampling weight'])
    print(f"{name}: {n_events} probable cases / {df.shape[0]} "
          f"(unweighted prevalence {unweighted_prev:.1%}, weighted {weighted_prev:.1%})")


Depression: 235 probable cases / 4887 (unweighted prevalence 4.8%, weighted 4.8%)
Anxiety: 221 probable cases / 4887 (unweighted prevalence 4.5%, weighted 4.6%)


## 3. Variable coding

Covariates enter as categorical (`C()`), matching how they were coded for the Notebook 4 GVIF check.
Reference categories are the substantively natural baseline in every case (youngest age band, no
education, no children, ...) except Wealth, where Richest is the reference so poorer categories read
directly as the hypothesized gradient.

Two variables that used to be considered "still open" are settled here:

- **Insurance** is not in the covariate list below. It's complete separation on both outcomes --
  zero probable-depression and zero probable-anxiety cases among the 16 insured women in the sample
  -- so its coefficient isn't a real estimate, just an artifact of a small, all-zero cell (see
  Section 5). Dropping it from the adjustment set changes the depression interaction LR test from
  p = 0.1816 to p = 0.1951 and the collapsed-Burden one from p = 0.0166 to p = 0.0180 -- close enough
  that nothing about the substantive story turns on this. Kept as a candidate for a future,
  Firth-penalized sensitivity model rather than the main pipeline; not pursued here.
- **Financial Decision-Making** and **IPV Attitude** are also not in the primary list. Section 21
  runs the full model again with both added, as an explicit sensitivity check, rather than folding
  them into the primary set or leaving the question open.

In [3]:
ref_levels = {
    'Socioeconomic Status': [5, 1, 2, 3, 4],   # ref = Richest
    'Education':            [0, 1, 2, 3],      # ref = No education
    'Occupation':           [0, 1],            # ref = No
    'Partner occupation':   [2, 1, 3],          # ref = Working
    'Age':                  [1, 2, 3],          # ref = 15-24
    'Division':             [3, 1, 2, 4, 5, 6, 7, 8],  # ref = Dhaka
    'Residence':            [1, 2],            # ref = Urban
    'Religion':             [1, 2],            # ref = Islam
    'Children':             [0, 1, 2, 3, 4],   # ref = No children
    'Family size':          [1, 2],            # ref = <5 members
    'Household Autonomy':   [0, 1, 2, 3],      # ref = No autonomy
    'Internet':             [0, 1, 2],         # ref = Never
    'Financial Decision-Making': [0, 1, 2],    # ref = Husband/other decides
    'IPV Attitude':         [0, 1, 2],         # ref = Rejects in all scenarios
}

covariate_cols = ['Education', 'Occupation', 'Partner occupation', 'Age', 'Division', 'Residence',
                   'Religion', 'Children', 'Family size', 'Household Autonomy', 'Internet']

def apply_ref_levels(df):
    df = df.copy()
    for col, order in ref_levels.items():
        df[col] = pd.Categorical(df[col], categories=order)
    return df

dep = apply_ref_levels(dep)
anx = apply_ref_levels(anx)
dep['Burden_num'] = dep['Cardiometabolic Burden'].astype(int)
anx['Burden_num'] = anx['Cardiometabolic Burden'].astype(int)
print(f'Primary covariate set: {len(covariate_cols)} variables.')


Primary covariate set: 11 variables.


## 4. Is the Wealth x Burden interaction even estimable?

Cross-tabulate Wealth x Burden before fitting anything. A fully categorical 5x4 interaction needs
every cell populated with enough events to identify a stable coefficient.

In [4]:
print("Wealth x Cardiometabolic Burden (uncollapsed, 4 levels):")
display(pd.crosstab(dep['Socioeconomic Status'], dep['Cardiometabolic Burden']))


Wealth x Cardiometabolic Burden (uncollapsed, 4 levels):


Cardiometabolic Burden,0,1,2,3
Socioeconomic Status,,,,
5,710,290,108,16
1,687,145,26,1
2,733,175,30,6
3,725,173,40,6
4,716,217,76,7


Wealth = Poorest x Burden = 3 has exactly 1 observation, so a fully categorical 5x4
interaction (12 free interaction parameters) isn't reliably estimable -- that one cell alone would
produce a coefficient driven by a single person. Two specifications instead:

- **Model 2 (primary):** Burden enters linearly (0-3) -- "one more cardiometabolic condition" as a
  constant step on the log-odds scale. A real assumption, but it pools information across all 4,887
  women instead of isolating it in sparse cells, and matches how `Figure-1` describes Burden ("score
  0-3").
- **Model 3 (sensitivity):** Burden collapsed to 3 levels (0 / 1 / 2+), which relaxes linearity while
  keeping every cell above n = 27.

In [5]:
dep['Burden_collapsed'] = dep['Cardiometabolic Burden'].astype(int).map({0: 0, 1: 1, 2: 2, 3: 2})
anx['Burden_collapsed'] = anx['Cardiometabolic Burden'].astype(int).map({0: 0, 1: 1, 2: 2, 3: 2})
dep['Burden_collapsed'] = pd.Categorical(dep['Burden_collapsed'], categories=[0, 1, 2])
anx['Burden_collapsed'] = pd.Categorical(anx['Burden_collapsed'], categories=[0, 1, 2])

print("Wealth x Burden_collapsed (0 / 1 / 2+) -- minimum cell size:")
ct = pd.crosstab(dep['Socioeconomic Status'], dep['Burden_collapsed'])
display(ct)
print("Minimum cell count:", ct.values.min())


Wealth x Burden_collapsed (0 / 1 / 2+) -- minimum cell size:


Burden_collapsed,0,1,2
Socioeconomic Status,,,
5,710,290,124
1,687,145,27
2,733,175,36
3,725,173,46
4,716,217,83


Minimum cell count: 27


That fixes the *sample-size* sparsity problem, but not necessarily the *event* sparsity
problem -- a cell can have plenty of women in it and still have zero of them screen positive, which
is complete separation all over again, just at the interaction-cell level instead of the
covariate-category level. Checked directly below (Section 6), after first checking the ordinary
covariates for the same issue.

## 5. Zero-event covariate categories

Any covariate category with zero outcome events is complete separation: that category's coefficient
will sit at a numerically meaningless extreme (0 or infinity) no matter how large the rest of the
sample is, with a standard error to match.

In [6]:
def check_zero_event_cells(df, outcome_col, covariates):
    flagged = []
    for v in covariates:
        ct = pd.crosstab(df[v], df[outcome_col])
        zero_rows = ct.index[(ct == 0).any(axis=1)]
        for level in zero_rows:
            flagged.append((v, level, int(ct.loc[level].sum())))
    return flagged

for label, df, outcome in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    flags = check_zero_event_cells(df, outcome, covariate_cols)
    print(f'{label}, primary covariate set:')
    for v, level, n in flags:
        print(f'  {v} = {level} (n={n}) -- complete separation')
    if not flags:
        print('  none')

print()
print("For reference, Insurance (excluded from the list above) on its own:")
for label, df, outcome in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    ins = pd.read_csv(f'../resources/{"depression" if label=="Depression" else "anxiety"}_dataset.csv')[['Insurance']]
    ct = pd.crosstab(ins['Insurance'], df[outcome].values)
    print(f'  {label}:'); print(ct.to_string().replace(chr(10), chr(10)+'  '))


Depression, primary covariate set:
  Partner occupation = 3 (n=8) -- complete separation
Anxiety, primary covariate set:
  none

For reference, Insurance (excluded from the list above) on its own:
  Depression:
col_0         0    1
  Insurance           
  0.0        4636  235
  1.0          16    0
  Anxiety:
col_0         0    1
  Insurance           
  0.0        4650  221
  1.0          16    0


Partner occupation = Don't know (n = 8) is complete separation for depression, but not for
anxiety -- it stays in the model either way (dropping it would throw away real information about the
other 8 women's covariate profile), but its own coefficient in the depression tables gets reported as
not estimable rather than as a spurious 0.000, starting in Section 8.

Insurance = Yes is complete separation for *both* outcomes (0 of 16 insured women screen positive on
either), which is a stronger case for exclusion than suppression: with only 2 levels total,
"suppress the Yes coefficient" would leave the variable contributing nothing but noise to every other
coefficient's standard error. That's why it's out of the covariate list rather than in-but-suppressed
like Partner occupation.

## 6. Zero-event Wealth x Burden cells

Same idea as Section 5, but for the interaction cells specifically -- a Wealth x Burden combination
can have zero events even when neither Wealth nor Burden does on its own.

In [7]:
def check_zero_event_interaction_cells(df, outcome_col, moderator_col, burden_col):
    flagged = []
    ct_n = pd.crosstab(df[moderator_col], df[burden_col])
    ct_events = pd.crosstab(df[moderator_col], df[burden_col], df[outcome_col], aggfunc='sum').fillna(0)
    for wealth in ct_n.index:
        for burden in ct_n.columns:
            n = ct_n.loc[wealth, burden]
            if n > 0 and ct_events.loc[wealth, burden] == 0:
                flagged.append((wealth, burden, int(n)))
    return flagged

for label, df, outcome in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    flags = check_zero_event_interaction_cells(df, outcome, 'Socioeconomic Status', 'Burden_collapsed')
    print(f'{label}, Wealth x Burden_collapsed cells with zero events:', flags if flags else 'none')


Depression, Wealth x Burden_collapsed cells with zero events: none
Anxiety, Wealth x Burden_collapsed cells with zero events: [(1, 2, 27)]


Anxiety has one: Wealth = Poorest x Burden_collapsed = 2+ (n = 27, 0 events). That specific
interaction term in Model 3 is complete separation the same way Insurance was -- it'll show up as
OR = 0.000 when the model is fit in Section 11, and gets suppressed the same way Partner
occupation = Don't know does for depression.

## 7. Model-fitting helpers

In [8]:
def fit_svy_logit(formula, data, weight_col='Sampling weight', cluster_col='PSU'):
    '''Survey-weighted logistic regression: weighted pseudo-likelihood point estimates,
    cluster-robust (PSU) sandwich standard errors. Stratum isn't incorporated -- see Section 1.'''
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model = smf.glm(formula=formula, data=data, family=sm.families.Binomial(),
                         var_weights=data[weight_col])
        res = model.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_col]})
        spec_warnings = [str(w.message) for w in caught if issubclass(w.category, sm.tools.sm_exceptions.SpecificationWarning)]
    if not res.converged:
        print("*** WARNING: model did not converge ***")
    return res, spec_warnings

def or_table(res, label=''):
    '''Tidy odds-ratio table with 95% CI from a fitted GLM result.'''
    params = res.params
    ci = res.conf_int()
    out = pd.DataFrame({
        'term': params.index,
        'OR': np.exp(params.values),
        'CI_low': np.exp(ci[0].values),
        'CI_high': np.exp(ci[1].values),
        'p': res.pvalues.values,
    })
    out = out[out['term'] != 'Intercept'].reset_index(drop=True)
    if label:
        out.insert(0, 'model', label)
    return out

def lr_test(res_restricted, res_full):
    '''Likelihood-ratio test for nested models (deviance difference ~ chi-square).'''
    df_diff = res_full.df_model - res_restricted.df_model
    stat = res_restricted.deviance - res_full.deviance
    p = stats.chi2.sf(stat, df_diff)
    return stat, df_diff, p

def suppress_nonidentifiable(df, patterns, note='not estimable (zero events in this cell)'):
    '''Blank out OR/CI/p for terms matching any of `patterns` (substrings of `term`) and say why,
    instead of reporting the spurious OR = 0.000 / infinite-SE numbers complete separation produces.'''
    df = df.copy()
    mask = df['term'].apply(lambda t: any(p in t for p in patterns))
    df['note'] = ''
    for col in ['OR', 'CI_low', 'CI_high', 'p']:
        df.loc[mask, col] = np.nan
    df.loc[mask, 'note'] = note
    return df


## 8. Depression models (primary covariate set)

- **Model 1:** main effects only (Wealth + Burden + 11 covariates)
- **Model 2 (primary):** + Wealth x Burden (linear) interaction
- **Model 1c / Model 3 (sensitivity):** same, with Burden collapsed to 0 / 1 / 2+

Model 1 and Model 3 are *not* directly comparable to each other -- they code the moderator
differently (linear vs. collapsed), so they're not nested. Model 1c exists specifically to give
Model 3 a valid nested comparator: same collapsed-Burden main effect, no interaction. (R's
`survey::anova.svyglm`, used in the companion R notebook, checks nesting and refuses a Model
1-vs-3 comparison outright -- worth keeping in mind if adapting this code elsewhere.)

In [9]:
covariate_formula = ' + '.join([f'C(Q("{c}"))' for c in covariate_cols])

f_dep_m1  = f'Depression_binary ~ C(Q("Socioeconomic Status")) + Burden_num + {covariate_formula}'
f_dep_m2  = f'Depression_binary ~ C(Q("Socioeconomic Status")) * Burden_num + {covariate_formula}'
f_dep_m1c = f'Depression_binary ~ C(Q("Socioeconomic Status")) + Q("Burden_collapsed") + {covariate_formula}'
f_dep_m3  = f'Depression_binary ~ C(Q("Socioeconomic Status")) * Q("Burden_collapsed") + {covariate_formula}'

res_dep_m1,  w1  = fit_svy_logit(f_dep_m1,  dep)
res_dep_m2,  w2  = fit_svy_logit(f_dep_m2,  dep)
res_dep_m1c, w1c = fit_svy_logit(f_dep_m1c, dep)
res_dep_m3,  w3  = fit_svy_logit(f_dep_m3,  dep)

print('Model 1 converged:', res_dep_m1.converged, '| Model 2 converged:', res_dep_m2.converged,
      '| Model 1c converged:', res_dep_m1c.converged, '| Model 3 converged:', res_dep_m3.converged)
if w2:
    print('statsmodels caveat (Model 2):', w2[0])


Model 1 converged: True | Model 2 converged: True | Model 1c converged: True | Model 3 converged: True
statsmodels caveat (Model 2): cov_type not fully supported with var_weights


Odds ratios below, with the Partner occupation = Don't know row suppressed per Section 5.

In [10]:
dep_m2_or = suppress_nonidentifiable(or_table(res_dep_m2, 'Depression M2'), ['Partner occupation"))[T.3]'])
print("Depression Model 2 (primary): Wealth x Burden odds ratios")
with pd.option_context('display.max_rows', 100):
    display(dep_m2_or.round(3))


Depression Model 2 (primary): Wealth x Burden odds ratios


,model,term,OR,CI_low,CI_high,p,note
0,Depression M2,"C(Q(""Socioeconomic Status""))[T.1]",1.607,0.823,3.140,0.165,
1,Depression M2,"C(Q(""Socioeconomic Status""))[T.2]",2.047,1.088,3.852,0.026,
2,Depression M2,"C(Q(""Socioeconomic Status""))[T.3]",1.786,0.915,3.484,0.089,
3,Depression M2,"C(Q(""Socioeconomic Status""))[T.4]",1.656,0.888,3.085,0.112,
4,Depression M2,"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.660,
5,Depression M2,"C(Q(""Education""))[T.2]",0.850,0.542,1.333,0.479,
6,Depression M2,"C(Q(""Education""))[T.3]",0.475,0.255,0.885,0.019,
7,Depression M2,"C(Q(""Occupation""))[T.1]",0.922,0.663,1.281,0.628,
8,Depression M2,"C(Q(""Partner occupation""))[T.1]",1.187,0.485,2.907,0.707,
9,Depression M2,"C(Q(""Partner occupation""))[T.3]",NaN,NaN,NaN,NaN,not estimable (zero events in this cell)


Once there's an interaction in the model, the `Socioeconomic Status` main-effect rows only
describe the wealth OR at `Burden_num = 0` -- the `...:Burden_num` rows are how much that OR
*changes* per extra burden point, not a standalone effect. Section 9 (simple slopes) turns this into
the actual wealth OR at each observed burden level, which is what should get reported and
interpreted, not the raw table above.

In [11]:
stat, df_diff, p = lr_test(res_dep_m1, res_dep_m2)
print(f"LR test, Depression Model 1 vs Model 2 (linear interaction): "
      f"chi2({df_diff:.0f}) = {stat:.2f}, p = {p:.4f}")

stat3, df_diff3, p3 = lr_test(res_dep_m1c, res_dep_m3)
print(f"LR test, Depression Model 1c vs Model 3 (collapsed-Burden interaction): "
      f"chi2({df_diff3:.0f}) = {stat3:.2f}, p = {p3:.4f}")


LR test, Depression Model 1 vs Model 2 (linear interaction): chi2(4) = 6.05, p = 0.1951
LR test, Depression Model 1c vs Model 3 (collapsed-Burden interaction): chi2(8) = 18.46, p = 0.0180


Model 3 comes out significant where Model 2 doesn't, which points at something specific
happening once burden reaches "2 or more" that a straight linear slope across 0-3 washes out.
Where, exactly, shows up in the interaction terms:

In [12]:
dep_m3_or = or_table(res_dep_m3, 'Depression M3')
interaction_rows = dep_m3_or[dep_m3_or['term'].str.contains(':')]
print("Depression Model 3: Wealth x Burden_collapsed interaction terms")
display(interaction_rows.round(3))


Depression Model 3: Wealth x Burden_collapsed interaction terms


,model,term,OR,CI_low,CI_high,p
33,Depression M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",1.043,0.280,3.888,0.950
34,Depression M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",1.150,0.387,3.419,0.802
35,Depression M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.693,0.166,2.888,0.615
36,Depression M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",2.040,0.569,7.318,0.274
37,Depression M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.610,0.104,3.584,0.584
38,Depression M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.150,0.017,1.357,0.091
39,Depression M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.505,0.103,2.484,0.401
40,Depression M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",0.029,0.003,0.256,0.001


Richer x Burden(2+) is the one doing the work: OR = 0.028, 95% CI [0.003, 0.250]. Read
together with the simple slopes below, the story is that the wealth gradient in depression risk --
Richer women look somewhat *more* likely to screen positive than the Richest at low burden -- flips
and widens sharply once cardiometabolic burden reaches two or more conditions. That reversal, not a
uniform "wealth protects less under burden" pattern, is what's driving the omnibus test.

One flag on that OR, though: it's built off a cell with real event sparsity (see Section 6 and the
E-value discussion in Section 17) -- worth treating as suggestive rather than a precise effect size
until it's cross-checked against R's design-based SE in Section 22.

### Full Model 2 covariate table (Depression)

In [13]:
with pd.option_context('display.max_rows', 100):
    display(dep_m2_or.round(3))


,model,term,OR,CI_low,CI_high,p,note
0,Depression M2,"C(Q(""Socioeconomic Status""))[T.1]",1.607,0.823,3.140,0.165,
1,Depression M2,"C(Q(""Socioeconomic Status""))[T.2]",2.047,1.088,3.852,0.026,
2,Depression M2,"C(Q(""Socioeconomic Status""))[T.3]",1.786,0.915,3.484,0.089,
3,Depression M2,"C(Q(""Socioeconomic Status""))[T.4]",1.656,0.888,3.085,0.112,
4,Depression M2,"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.660,
5,Depression M2,"C(Q(""Education""))[T.2]",0.850,0.542,1.333,0.479,
6,Depression M2,"C(Q(""Education""))[T.3]",0.475,0.255,0.885,0.019,
7,Depression M2,"C(Q(""Occupation""))[T.1]",0.922,0.663,1.281,0.628,
8,Depression M2,"C(Q(""Partner occupation""))[T.1]",1.187,0.485,2.907,0.707,
9,Depression M2,"C(Q(""Partner occupation""))[T.3]",NaN,NaN,NaN,NaN,not estimable (zero events in this cell)


## 9. Simple slopes: wealth OR at each level of cardiometabolic burden (Depression)

For Wealth level *k* vs. Richest, the log-odds ratio at `Burden_num = b` is
`beta_k + b * beta_(k:Burden)`. Variance comes from the delta method using Model 2's own covariance
matrix, so these are exact linear combinations of Model 2's parameters, not a separately fit
model.

In [14]:
def simple_slopes(res, wealth_col_prefix, burden_var, burden_values, wealth_levels):
    cov = res.cov_params()
    params = res.params
    rows = []
    for level in wealth_levels:
        main_term = f'{wealth_col_prefix}[T.{level}]'
        int_term = f'{wealth_col_prefix}[T.{level}]:{burden_var}'
        if main_term not in params.index:
            continue
        for b in burden_values:
            beta = params[main_term] + b * params.get(int_term, 0.0)
            var = cov.loc[main_term, main_term]
            if int_term in cov.index:
                var += (b ** 2) * cov.loc[int_term, int_term]
                var += 2 * b * cov.loc[main_term, int_term]
            se = np.sqrt(var)
            or_ = np.exp(beta)
            lo, hi = np.exp(beta - 1.96 * se), np.exp(beta + 1.96 * se)
            z = beta / se
            p = 2 * (1 - stats.norm.cdf(abs(z)))
            rows.append({'Wealth (vs Richest)': level, 'Burden': b, 'OR': or_, 'CI_low': lo, 'CI_high': hi, 'p': p})
    return pd.DataFrame(rows)

wealth_col = 'C(Q("Socioeconomic Status"))'
dep_slopes = simple_slopes(res_dep_m2, wealth_col, 'Burden_num', [0, 1, 2, 3], [1, 2, 3, 4])
print("Depression: Wealth OR (vs Richest) at each Cardiometabolic Burden level")
display(dep_slopes.round(3))


Depression: Wealth OR (vs Richest) at each Cardiometabolic Burden level


,Wealth (vs Richest),Burden,OR,CI_low,CI_high,p
0,1,0,1.607,0.823,3.140,0.165
1,1,1,1.041,0.471,2.303,0.921
2,1,2,0.675,0.153,2.966,0.602
3,1,3,0.437,0.045,4.217,0.474
4,2,0,2.047,1.088,3.852,0.026
5,2,1,1.006,0.506,1.996,0.987
6,2,2,0.494,0.143,1.705,0.265
7,2,3,0.243,0.036,1.616,0.143
8,3,0,1.786,0.915,3.484,0.089
9,3,1,1.098,0.510,2.366,0.811


Each row is the odds of probable depression for a given wealth category relative to
Richest, at a specific burden level, covariates held fixed. If an OR moves further from 1 as burden
climbs, that's effect modification -- the wealth gap in depression risk widens with more
cardiometabolic burden. Flat across burden levels means no real modification for that category,
whatever the raw wealth main effect looks like. Read this table only if the omnibus test in the
previous section is significant -- otherwise a "significant-looking" individual row is likely just
multiple-comparison noise.

## 10. Stratified cluster bootstrap (Depression, Model 2)

Resamples PSUs with replacement *within stratum*, keeping the number of PSUs per stratum fixed --
closer to BDHS's actual two-stage stratified design than the plain cluster-robust sandwich above,
which ignores `Stratum` entirely. 300 replicates (compute-time tradeoff for the 36-parameter model in
this environment; bump to 1,000+ for a submission-grade bootstrap if it's worth the wait).

In [15]:
def stratified_cluster_bootstrap(df, formula, n_boot=300, seed=42, max_abs_coef=15):
    # Fast index-based resampling: precompute (Stratum, PSU) -> row-position array once, then build
    # each replicate by concatenating position arrays.
    # A sparse cell (Partner occupation = Don't know, n=8) occasionally lands at zero count in a
    # resampled replicate, producing a rank-deficient design matrix / degenerate fit. Two guards:
    # skip a replicate up front if its design matrix is rank-deficient, and as a backstop, drop any
    # replicate that still produces an implausibly extreme coefficient (|coef| > max_abs_coef, i.e.
    # OR > ~3.3 million).
    df = df.reset_index(drop=True)
    group_indices = df.groupby(['Stratum', 'PSU']).indices
    strata = df['Stratum'].unique()
    psu_by_stratum = {s: df.loc[df['Stratum'] == s, 'PSU'].unique() for s in strata}
    rng = np.random.default_rng(seed)
    boot_params = []
    n_rank_deficient = 0
    for _ in range(n_boot):
        idx_parts = []
        for s in strata:
            psus = psu_by_stratum[s]
            chosen = rng.choice(psus, size=len(psus), replace=True)
            idx_parts.extend(group_indices[(s, psu)] for psu in chosen)
        boot_df = df.iloc[np.concatenate(idx_parts)].reset_index(drop=True)
        try:
            model = smf.glm(formula=formula, data=boot_df, family=sm.families.Binomial(),
                             var_weights=boot_df['Sampling weight'])
            if np.linalg.matrix_rank(model.exog) < model.exog.shape[1]:
                n_rank_deficient += 1
                continue
            res = model.fit()
            if res.converged:
                boot_params.append(res.params)
        except Exception:
            continue
    print(f"  (excluded {n_rank_deficient} rank-deficient replicates out of {n_boot} attempted; "
          f"{len(boot_params)} whole-replicate fits retained)")
    boot_df_params = pd.DataFrame(boot_params)
    # Per-term masking rather than per-replicate exclusion: a chronically sparse term (Partner
    # occupation = Don't know, n=8) would wipe out nearly every replicate if the whole row were
    # dropped for it. Masking only the offending cell keeps the other ~35 well-behaved terms' full
    # 300 replicates.
    n_usable_per_term = boot_df_params.notna().sum() - (boot_df_params.abs() > max_abs_coef).sum()
    boot_df_params_masked = boot_df_params.mask(boot_df_params.abs() > max_abs_coef)
    return boot_df_params_masked, n_usable_per_term

boot_dep, n_usable_dep = stratified_cluster_bootstrap(dep, f_dep_m2, n_boot=300)

boot_ci = boot_dep.quantile([0.025, 0.975]).T
boot_ci.columns = ['boot_CI_low_logodds', 'boot_CI_high_logodds']
boot_ci['boot_OR_low'] = np.exp(boot_ci['boot_CI_low_logodds'])
boot_ci['boot_OR_high'] = np.exp(boot_ci['boot_CI_high_logodds'])
boot_ci = boot_ci.drop(columns=['boot_CI_low_logodds', 'boot_CI_high_logodds'])
boot_ci['n_usable_reps'] = n_usable_dep

compare = dep_m2_or.set_index('term')[['OR', 'CI_low', 'CI_high']].join(boot_ci, how='right').drop(index='Intercept', errors='ignore')
print("\nComparison: cluster-robust CI vs. stratified cluster-bootstrap CI")
print("(n_usable_reps well below 300 flags a term too sparse for the bootstrap CI to be trustworthy --")
print(" defer to the statsmodels or R/Stata CI for that term instead)")
with pd.option_context('display.max_rows', 100):
    display(compare.round(3))


  (excluded 1 rank-deficient replicates out of 300 attempted; 299 whole-replicate fits retained)

Comparison: cluster-robust CI vs. stratified cluster-bootstrap CI
(n_usable_reps well below 300 flags a term too sparse for the bootstrap CI to be trustworthy --
 defer to the statsmodels or R/Stata CI for that term instead)


,OR,CI_low,CI_high,boot_OR_low,boot_OR_high,n_usable_reps
"C(Q(""Socioeconomic Status""))[T.1]",1.607,0.823,3.140,0.810,3.641,299
"C(Q(""Socioeconomic Status""))[T.2]",2.047,1.088,3.852,1.136,4.181,299
"C(Q(""Socioeconomic Status""))[T.3]",1.786,0.915,3.484,0.942,3.909,299
"C(Q(""Socioeconomic Status""))[T.4]",1.656,0.888,3.085,0.901,3.247,299
"C(Q(""Education""))[T.1]",0.910,0.597,1.387,0.607,1.404,299
"C(Q(""Education""))[T.2]",0.850,0.542,1.333,0.556,1.301,299
"C(Q(""Education""))[T.3]",0.475,0.255,0.885,0.248,0.801,299
"C(Q(""Occupation""))[T.1]",0.922,0.663,1.281,0.671,1.304,299
"C(Q(""Partner occupation""))[T.1]",1.187,0.485,2.907,0.283,2.694,299
"C(Q(""Partner occupation""))[T.3]",NaN,NaN,NaN,NaN,NaN,0


Cluster-robust and bootstrap CIs land in the same place and don't disagree on which side of
1.0 anything sits, which is reassurance the cluster-only sandwich isn't badly misleading here --
still a Python-side cross-check, though, not a substitute for the R replication in Section 22.

## 11. Anxiety models (primary covariate set)

Same structure and covariate set as Section 8. The Model 3 interaction table below has its own
suppressed cell this time -- Wealth = Poorest x Burden_collapsed = 2+ is zero-event for anxiety (see
Section 6), so that term gets the same "not estimable" treatment Partner occupation got for
depression.

In [16]:
f_anx_m1  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) + Burden_num + {covariate_formula}'
f_anx_m2  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) * Burden_num + {covariate_formula}'
f_anx_m1c = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) + Q("Burden_collapsed") + {covariate_formula}'
f_anx_m3  = f'Anxiety_binary ~ C(Q("Socioeconomic Status")) * Q("Burden_collapsed") + {covariate_formula}'

res_anx_m1,  _   = fit_svy_logit(f_anx_m1,  anx)
res_anx_m2,  w2a = fit_svy_logit(f_anx_m2,  anx)
res_anx_m1c, _   = fit_svy_logit(f_anx_m1c, anx)
res_anx_m3,  _   = fit_svy_logit(f_anx_m3,  anx)

print('Model 1 converged:', res_anx_m1.converged, '| Model 2 converged:', res_anx_m2.converged,
      '| Model 1c converged:', res_anx_m1c.converged, '| Model 3 converged:', res_anx_m3.converged)

anx_m2_or = or_table(res_anx_m2, 'Anxiety M2')
print("\nAnxiety Model 2 (primary): Wealth x Burden odds ratios")
with pd.option_context('display.max_rows', 100):
    display(anx_m2_or.round(3))


Model 1 converged: True | Model 2 converged: True | Model 1c converged: True | Model 3 converged: True

Anxiety Model 2 (primary): Wealth x Burden odds ratios


,model,term,OR,CI_low,CI_high,p
0,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.1]",1.006,0.464,2.181,0.987
1,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.2]",1.305,0.675,2.521,0.429
2,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.3]",1.131,0.558,2.295,0.733
3,Anxiety M2,"C(Q(""Socioeconomic Status""))[T.4]",1.342,0.716,2.516,0.358
4,Anxiety M2,"C(Q(""Education""))[T.1]",0.854,0.544,1.342,0.495
5,Anxiety M2,"C(Q(""Education""))[T.2]",0.938,0.613,1.437,0.769
6,Anxiety M2,"C(Q(""Education""))[T.3]",0.474,0.222,1.012,0.054
7,Anxiety M2,"C(Q(""Occupation""))[T.1]",0.921,0.653,1.297,0.636
8,Anxiety M2,"C(Q(""Partner occupation""))[T.1]",1.342,0.653,2.761,0.424
9,Anxiety M2,"C(Q(""Partner occupation""))[T.3]",1.472,0.164,13.231,0.730


In [17]:
stat_a, df_diff_a, p_a = lr_test(res_anx_m1, res_anx_m2)
print(f"LR test, Anxiety Model 1 vs Model 2 (linear interaction): "
      f"chi2({df_diff_a:.0f}) = {stat_a:.2f}, p = {p_a:.4f}")

stat_a3, df_diff_a3, p_a3 = lr_test(res_anx_m1c, res_anx_m3)
print(f"LR test, Anxiety Model 1c vs Model 3 (collapsed-Burden interaction): "
      f"chi2({df_diff_a3:.0f}) = {stat_a3:.2f}, p = {p_a3:.4f}")


LR test, Anxiety Model 1 vs Model 2 (linear interaction): chi2(4) = 5.66, p = 0.2263
LR test, Anxiety Model 1c vs Model 3 (collapsed-Burden interaction): chi2(8) = 8.31, p = 0.4036


Neither test is anywhere near significant for anxiety, in either parameterization. Shown
below mostly for completeness / symmetry with the depression result, with the zero-event cell
suppressed rather than reported as OR = 0.000.

In [18]:
anx_m3_or = suppress_nonidentifiable(or_table(res_anx_m3, 'Anxiety M3'), ['Status"))[T.1]:Q("Burden_collapsed")[T.2]'])
interaction_rows_a = anx_m3_or[anx_m3_or['term'].str.contains(':')]
print("Anxiety Model 3: Wealth x Burden_collapsed interaction terms")
display(interaction_rows_a.round(3))


Anxiety Model 3: Wealth x Burden_collapsed interaction terms


,model,term,OR,CI_low,CI_high,p,note
33,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.483,0.130,1.797,0.278,
34,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.419,0.124,1.412,0.160,
35,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.922,0.306,2.774,0.885,
36,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",1.185,0.414,3.396,0.752,
37,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",NaN,NaN,NaN,NaN,not estimable (zero events in this cell)
38,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.224,0.021,2.399,0.216,
39,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.957,0.106,8.640,0.969,
40,Anxiety M3,"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",0.423,0.081,2.201,0.307,


## 12. Simple slopes and bootstrap (Anxiety)

Same delta-method / bootstrap machinery as Sections 9-10, run on the anxiety Model 2.

In [19]:
anx_slopes = simple_slopes(res_anx_m2, wealth_col, 'Burden_num', [0, 1, 2, 3], [1, 2, 3, 4])
print("Anxiety: Wealth OR (vs Richest) at each Cardiometabolic Burden level")
display(anx_slopes.round(3))


Anxiety: Wealth OR (vs Richest) at each Cardiometabolic Burden level


,Wealth (vs Richest),Burden,OR,CI_low,CI_high,p
0,1,0,1.006,0.464,2.181,0.987
1,1,1,0.455,0.166,1.244,0.125
2,1,2,0.206,0.031,1.384,0.104
3,1,3,0.093,0.005,1.708,0.110
4,2,0,1.305,0.675,2.521,0.429
5,2,1,0.602,0.264,1.372,0.227
6,2,2,0.278,0.057,1.343,0.111
7,2,3,0.128,0.011,1.439,0.096
8,3,0,1.131,0.558,2.295,0.733
9,3,1,1.145,0.551,2.381,0.716


In [20]:
boot_anx, n_usable_anx = stratified_cluster_bootstrap(anx, f_anx_m2, n_boot=300)

boot_ci_a = boot_anx.quantile([0.025, 0.975]).T
boot_ci_a.columns = ['boot_CI_low_logodds', 'boot_CI_high_logodds']
boot_ci_a['boot_OR_low'] = np.exp(boot_ci_a['boot_CI_low_logodds'])
boot_ci_a['boot_OR_high'] = np.exp(boot_ci_a['boot_CI_high_logodds'])
boot_ci_a = boot_ci_a.drop(columns=['boot_CI_low_logodds', 'boot_CI_high_logodds'])
boot_ci_a['n_usable_reps'] = n_usable_anx

compare_a = anx_m2_or.set_index('term')[['OR', 'CI_low', 'CI_high']].join(boot_ci_a, how='right').drop(index='Intercept', errors='ignore')
print("\nComparison: cluster-robust CI vs. stratified cluster-bootstrap CI (Anxiety)")
with pd.option_context('display.max_rows', 100):
    display(compare_a.round(3))


  (excluded 1 rank-deficient replicates out of 300 attempted; 299 whole-replicate fits retained)

Comparison: cluster-robust CI vs. stratified cluster-bootstrap CI (Anxiety)


,OR,CI_low,CI_high,boot_OR_low,boot_OR_high,n_usable_reps
"C(Q(""Socioeconomic Status""))[T.1]",1.006,0.464,2.181,0.418,2.370,299
"C(Q(""Socioeconomic Status""))[T.2]",1.305,0.675,2.521,0.674,2.732,299
"C(Q(""Socioeconomic Status""))[T.3]",1.131,0.558,2.295,0.552,2.386,299
"C(Q(""Socioeconomic Status""))[T.4]",1.342,0.716,2.516,0.665,2.560,299
"C(Q(""Education""))[T.1]",0.854,0.544,1.342,0.506,1.317,299
"C(Q(""Education""))[T.2]",0.938,0.613,1.437,0.615,1.463,299
"C(Q(""Education""))[T.3]",0.474,0.222,1.012,0.201,0.948,299
"C(Q(""Occupation""))[T.1]",0.921,0.653,1.297,0.643,1.248,299
"C(Q(""Partner occupation""))[T.1]",1.342,0.653,2.761,0.497,2.494,299
"C(Q(""Partner occupation""))[T.3]",1.472,0.164,13.231,0.615,11.086,190


## 13. Combined summary: Wealth x Burden interaction, both outcomes

In [21]:
summary_rows = []
for outcome, res_m1, res_m2, lr_stat, lr_df, lr_p, res_m1c, res_m3, lr_stat3, lr_df3, lr_p3 in [
    ('Depression', res_dep_m1, res_dep_m2, stat, df_diff, p, res_dep_m1c, res_dep_m3, stat3, df_diff3, p3),
    ('Anxiety', res_anx_m1, res_anx_m2, stat_a, df_diff_a, p_a, res_anx_m1c, res_anx_m3, stat_a3, df_diff_a3, p_a3),
]:
    summary_rows.append({
        'Outcome': outcome,
        'N': int(res_m2.nobs),
        'Events': int(res_m1.model.endog.sum()),
        'M2 AIC': round(res_m2.aic, 1),
        'Linear-int. chi2': round(lr_stat, 2), 'df': int(lr_df), 'Linear-int. p': round(lr_p, 4),
        'Collapsed-int. chi2': round(lr_stat3, 2), 'df ': int(lr_df3), 'Collapsed-int. p': round(lr_p3, 4),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,Outcome,N,Events,M2 AIC,Linear-int. chi2,df,Linear-int. p,Collapsed-int. chi2,df,Collapsed-int. p
0,Depression,4887,235,1879.5,6.05,4,0.1951,18.46,8,0.0180
1,Anxiety,4887,221,1808.1,5.66,4,0.2263,8.31,8,0.4036


Depression's collapsed-Burden interaction is the one finding here (p = 0.018, uncorrected)
-- everything else, including depression's own linear-interaction test, lands well above 0.05.
Section 18 puts that single p-value through a multiple-testing correction across all four tests run
in this section; Section 21 checks whether it survives adding two more covariates.

## 14. Goodness of fit

A Hosmer-Lemeshow-style check on Model 2 for both outcomes: group women into deciles of predicted
risk, compare observed vs. expected event counts (weighted) within each decile. This is a
design-naive version -- it doesn't account for clustering or stratification the way a proper
survey-adjusted GOF test would -- so it's exploratory here, not the number to cite. Stata's
`estat gof` after `svy: logit` (Archer & Lemeshow 2006) is the validated design-based version;
that line is added to the Stata block in Section 22.

In [22]:
def weighted_hl_test(res, data, outcome_col, weight_col='Sampling weight', g=10):
    yhat = res.predict(data)
    d = pd.DataFrame({'y': data[outcome_col].values, 'yhat': yhat.values, 'w': data[weight_col].values})
    d['decile'] = pd.qcut(d['yhat'], q=g, duplicates='drop')
    grp = d.groupby('decile', observed=True)
    obs1 = grp.apply(lambda x: np.sum(x['y'] * x['w']), include_groups=False)
    exp1 = grp.apply(lambda x: np.sum(x['yhat'] * x['w']), include_groups=False)
    n_w = grp['w'].sum()
    obs0, exp0 = n_w - obs1, n_w - exp1
    stat = np.sum((obs1 - exp1) ** 2 / exp1 + (obs0 - exp0) ** 2 / exp0)
    dfree = len(grp) - 2
    return stat, dfree, stats.chi2.sf(stat, dfree)

for label, df, ycol, res in [('Depression', dep, 'Depression_binary', res_dep_m2),
                              ('Anxiety', anx, 'Anxiety_binary', res_anx_m2)]:
    stat_g, dfree_g, p_g = weighted_hl_test(res, df, ycol)
    verdict = 'some evidence of poor fit' if p_g < 0.05 else 'no evidence of poor fit'
    print(f"{label} M2: chi2={stat_g:.2f}, df={dfree_g}, p={p_g:.4f} -> {verdict}")


Depression M2: chi2=9.53, df=8, p=0.2993 -> no evidence of poor fit
Anxiety M2: chi2=12.02, df=8, p=0.1501 -> no evidence of poor fit


Neither outcome shows evidence of poor fit by this check. Reassuring, but again: confirm with
`estat gof` before leaning on it for the manuscript.

## 15. Linearity of the Burden term

Model 1 treats Burden as linear (0-3). Refit with Burden as an unordered 4-level factor instead and
compare via LR test -- if the categorical version fits meaningfully better, the linear-Burden
assumption used throughout Models 1/2 is worth reconsidering.

In [23]:
for label, df, ycol, res_m1 in [('Depression', dep, 'Depression_binary', res_dep_m1),
                                       ('Anxiety', anx, 'Anxiety_binary', res_anx_m1)]:
    df = df.copy()
    df['Burden_cat4'] = pd.Categorical(df['Cardiometabolic Burden'].astype(int), categories=[0, 1, 2, 3])
    f_cat4 = f'{ycol} ~ C(Q("Socioeconomic Status")) + C(Q("Burden_cat4")) + {covariate_formula}'
    res_cat4, _ = fit_svy_logit(f_cat4, df)
    stat_l, dfd_l, p_l = lr_test(res_m1, res_cat4)
    verdict = 'evidence against linearity' if p_l < 0.05 else 'no evidence against linearity'
    print(f"{label}: chi2({dfd_l:.0f}) = {stat_l:.2f}, p = {p_l:.4f} -> {verdict}")


Depression: chi2(2) = 2.51, p = 0.2850 -> no evidence against linearity


Anxiety: chi2(2) = 0.54, p = 0.7643 -> no evidence against linearity


No evidence against linearity for either outcome -- the categorical Burden coding doesn't fit
appreciably better than the linear one, which is some support for using linear Burden as the primary
specification in Model 2 rather than just a convenience for the sparse-cell problem in Section 4.

## 16. Events per variable

A rough rule of thumb for stable logistic-regression coefficients is >= 10 events per estimated
parameter; below that, standard errors and p-values start getting unreliable in ways that don't
necessarily announce themselves.

In [24]:
epv_rows = []
for label, res_m1, res_m2 in [('Depression', res_dep_m1, res_dep_m2), ('Anxiety', res_anx_m1, res_anx_m2)]:
    n_events = int(res_m1.model.endog.sum())
    epv_rows.append({'Outcome': label, 'Model': 'M1 (main effects)', 'Events': n_events,
                      'Params': int(res_m1.df_model), 'EPV': round(n_events / res_m1.df_model, 2)})
    epv_rows.append({'Outcome': label, 'Model': 'M2 (+ interaction)', 'Events': n_events,
                      'Params': int(res_m2.df_model), 'EPV': round(n_events / res_m2.df_model, 2)})
display(pd.DataFrame(epv_rows))


,Outcome,Model,Events,Params,EPV
0,Depression,M1 (main effects),235,32,7.34
1,Depression,M2 (+ interaction),235,36,6.53
2,Anxiety,M1 (main effects),221,32,6.91
3,Anxiety,M2 (+ interaction),221,36,6.14


Model 2 sits at roughly 6 events per parameter for both outcomes -- below the conventional
10-EPV rule of thumb. That's a direct consequence of a 36-parameter model against 235 (depression) /
221 (anxiety) events, not a modeling mistake, but it's a real limitation: individual coefficient SEs,
especially for the interaction terms, are less trustworthy than the point estimates. This is exactly
why Section 9's omnibus LR test -- not individual simple-slope p-values -- is treated as the primary
evidence for or against effect modification. Section 21 shows how EPV moves (down, since two more
parameters get added) once Financial Decision-Making and IPV Attitude join the covariate set.

## 17. Multiple testing

Section 13's summary table reports four LR tests (2 outcomes x 2 Burden parameterizations). Only
one -- depression, collapsed Burden -- clears p < 0.05 uncorrected. Two ways to frame the correction,
depending on how "the family of tests" is defined:

In [25]:
primary = [p, p_a]                    # the two linear-interaction tests, the pre-specified primary analysis
all_four = [p, p_a, p3, p_a3]          # + the two collapsed-Burden sensitivity tests

print("Framing A -- primary analysis only (2 linear-interaction tests):")
print(f"  raw p:         {[round(x, 4) for x in primary]}")
print(f"  Bonferroni:    {[round(x, 4) for x in multipletests(primary, method='bonferroni')[1]]}")
print(f"  BH-FDR:        {[round(x, 4) for x in multipletests(primary, method='fdr_bh')[1]]}")
print()
print("Framing B -- all four tests run in Section 13 (both parameterizations, both outcomes):")
print(f"  raw p:         {[round(x, 4) for x in all_four]}")
print(f"  Bonferroni:    {[round(x, 4) for x in multipletests(all_four, method='bonferroni')[1]]}")
print(f"  BH-FDR:        {[round(x, 4) for x in multipletests(all_four, method='fdr_bh')[1]]}")


Framing A -- primary analysis only (2 linear-interaction tests):
  raw p:         [np.float64(0.1951), np.float64(0.2263)]
  Bonferroni:    [np.float64(0.3902), np.float64(0.4525)]
  BH-FDR:        [np.float64(0.2263), np.float64(0.2263)]

Framing B -- all four tests run in Section 13 (both parameterizations, both outcomes):
  raw p:         [np.float64(0.1951), np.float64(0.2263), np.float64(0.018), np.float64(0.4036)]
  Bonferroni:    [np.float64(0.7803), np.float64(0.9051), np.float64(0.0721), np.float64(1.0)]
  BH-FDR:        [np.float64(0.3017), np.float64(0.3017), np.float64(0.0721), np.float64(0.4036)]


Framing A barely changes anything, since neither linear-interaction test was close to
significant to begin with. Framing B is the honest one, though -- the collapsed-Burden coding in
Model 3 was chosen *because* the linear version didn't turn up much (Section 8), which makes it a
post-hoc sensitivity check, not a second pre-registered primary test, and it should be corrected
alongside the others rather than reported as if it stood alone. Under Framing B, depression's
collapsed-interaction p-value (0.018 raw) moves to roughly 0.07 either way -- no longer conventionally
significant. That doesn't erase the finding, but it does mean it should be reported as
"suggestive, not confirmed" rather than as a confirmed effect-modification result.

## 18. E-values

How strong would an unmeasured confounder's association with *both* wealth and the outcome need to
be to fully explain away an observed OR, on the risk-ratio scale (VanderWeele & Ding, 2017,
*Ann Intern Med*)? Computed for the point estimate and for the confidence limit closer to the null
(the more conservative of the two).

In [26]:
def e_value(estimate, ci_bound=None):
    def _rr(x):
        return x if x >= 1 else 1 / x
    rr = _rr(estimate)
    ev_point = rr + np.sqrt(rr * (rr - 1))
    ev_ci = None
    if ci_bound is not None:
        rr_ci = _rr(ci_bound)
        ev_ci = 1.0 if rr_ci <= 1 else rr_ci + np.sqrt(rr_ci * (rr_ci - 1))
    return ev_point, ev_ci

ev_rows = []

poorer_dep = dep_slopes[(dep_slopes['Wealth (vs Richest)'] == 2) & (dep_slopes['Burden'] == 0)].iloc[0]
ev_p, ev_c = e_value(poorer_dep.OR, poorer_dep.CI_low)
ev_rows.append({'Estimate': 'Wealth=Poorer vs Richest, Depression @ Burden=0', 'OR': round(poorer_dep.OR, 3),
                 '95% CI': f'[{poorer_dep.CI_low:.3f}, {poorer_dep.CI_high:.3f}]',
                 'E-value (point)': round(ev_p, 2), 'E-value (CI limit)': round(ev_c, 2)})

age_row = dep_m2_or[dep_m2_or['term'].str.contains('Age')].iloc[1]  # T.3 = 35-49 vs 15-24
ev_p, ev_c = e_value(age_row.OR, age_row.CI_low)
ev_rows.append({'Estimate': 'Age 35-49 vs 15-24, Depression', 'OR': round(age_row.OR, 3),
                 '95% CI': f'[{age_row.CI_low:.3f}, {age_row.CI_high:.3f}]',
                 'E-value (point)': round(ev_p, 2), 'E-value (CI limit)': round(ev_c, 2)})

richer_int = interaction_rows[interaction_rows['term'].str.contains(r'T\.4.*Burden_collapsed.*T\.2')].iloc[0]
ev_p, ev_c = e_value(richer_int.OR, richer_int.CI_high)  # OR < 1 -> CI_high is the bound closer to null
ev_rows.append({'Estimate': 'Wealth=Richer x Burden(2+), Depression interaction', 'OR': round(richer_int.OR, 3),
                 '95% CI': f'[{richer_int.CI_low:.3f}, {richer_int.CI_high:.3f}]',
                 'E-value (point)': round(ev_p, 2), 'E-value (CI limit)': round(ev_c, 2)})

display(pd.DataFrame(ev_rows))


,Estimate,OR,95% CI,E-value (point),E-value (CI limit)
0,"Wealth=Poorer vs Richest, Depression @ Burden=0",2.047,"[1.088, 3.852]",3.51,1.40
1,"Age 35-49 vs 15-24, Depression",2.031,"[1.186, 3.480]",3.48,1.65
2,"Wealth=Richer x Burden(2+), Depression interac...",0.029,"[0.003, 0.256]",68.34,7.27


The two main-effect estimates (wealth, age) would take a moderately strong unmeasured
confounder to explain away -- an E-value around 3.5 means a confounder would need to be associated
with roughly a 3.5-fold risk-ratio with *both* wealth and depression, above and beyond everything
already adjusted for, to fully account for the association. Plausible confounders like unmeasured
chronic stress or extended-family conflict could plausibly reach that, so "moderately robust," not
"bulletproof."

The interaction term's own E-value (68 at the point estimate) looks enormous, but that's mostly an
artifact of how extreme the point estimate is on a sparse cell, not evidence of unusual robustness --
the CI-limit version (7.3) is the more honest read, and even that inherits all the sparse-cell
caveats from Section 8. Treat this one as illustrative, not as a claim that the interaction is
unconfoundable.

## 19. Outcome-cutoff sensitivity: PHQ-9 >= 15

Section 2 used the standard PHQ-9 >= 10 "moderate or worse" cutoff. Refitting at a stricter >= 15
("moderately severe or worse") checks whether the interaction conclusion is an artifact of exactly
where that line gets drawn.

In [27]:
dep_severe = dep.copy()
dep_severe['Depression_severe'] = (dep_severe['Depression'] >= 3).astype(int)
n_events_10 = int(dep['Depression_binary'].sum())
n_events_15 = int(dep_severe['Depression_severe'].sum())
print(f"Probable cases at >=10: {n_events_10}  |  at >=15: {n_events_15}")

f_m1_sev = f'Depression_severe ~ C(Q("Socioeconomic Status")) + Burden_num + {covariate_formula}'
f_m2_sev = f'Depression_severe ~ C(Q("Socioeconomic Status")) * Burden_num + {covariate_formula}'
res_m1_sev, _ = fit_svy_logit(f_m1_sev, dep_severe)
res_m2_sev, _ = fit_svy_logit(f_m2_sev, dep_severe)
stat_sev, dfd_sev, p_sev = lr_test(res_m1_sev, res_m2_sev)
epv_sev = n_events_15 / int(res_m2_sev.df_model)
print(f"M1 vs M2 LR test at >=15: chi2({dfd_sev:.0f}) = {stat_sev:.2f}, p = {p_sev:.4f}  (EPV = {n_events_15}/{int(res_m2_sev.df_model)} = {epv_sev:.2f})")


Probable cases at >=10: 235  |  at >=15: 62


M1 vs M2 LR test at >=15: chi2(4) = 7.94, p = 0.0939  (EPV = 62/36 = 1.72)


Only 62 women clear the stricter cutoff, which drags EPV down to about 1.7 -- far too sparse
to draw any real conclusion from this specific fit one way or the other. The right reading isn't
"the interaction disappears at >=15" (it might just be underpowered here), it's that >=10 is the
cutoff doing the actual analytical work in this paper, and that should be stated plainly rather than
implied.

## 20. Sampling weight range

Extreme weights can single-handedly distort a survey-weighted estimate. Matches the Kish design-
effect check already done in Notebook 4 -- restated here since it's directly relevant to how much to
trust the weighted models in this notebook specifically.

In [28]:
w = dep['Sampling weight']
print(f"min = {w.min():.4f}, max = {w.max():.4f}, mean = {w.mean():.4f}, max/min ratio = {w.max()/w.min():.1f}")


min = 0.0913, max = 3.8917, mean = 0.9928, max/min ratio = 42.6


A 43-fold spread between the smallest and largest weight is on the wide side, but consistent
with a normalized DHS sampling weight (no single case is running away with the analysis by itself) --
Notebook 4's Kish design effect for this same weight variable already covers whether this level of
variability is inflating variance meaningfully. No trimming or calibration applied here, consistent
with that earlier decision.

## 21. Influential observations

Cook's distance from the (non-survey) GLM fit, as a check for individual women disproportionately
driving Model 2's depression coefficients -- separate from the design-based SE question, this is
about whether a handful of data points are doing more work than they should.

In [29]:
infl = res_dep_m2.get_influence()
cooks_d = infl.cooks_distance[0]
threshold = 4 / len(dep)
n_flagged = int((cooks_d > threshold).sum())
print(f"Max Cook's D: {cooks_d.max():.4f}")
print(f"Cases exceeding the 4/n threshold ({threshold:.5f}): {n_flagged} / {len(dep)}")

top5 = np.argsort(cooks_d)[-5:][::-1]
print("\nTop 5 by Cook's D:")
display(dep.iloc[top5][['Socioeconomic Status', 'Cardiometabolic Burden', 'Depression_binary']].assign(cooks_d=cooks_d[top5]))


Max Cook's D: 0.0444
Cases exceeding the 4/n threshold (0.00082): 214 / 4887

Top 5 by Cook's D:


,Socioeconomic Status,Cardiometabolic Burden,Depression_binary,cooks_d
1455,5,2,1,0.044393
1347,5,2,1,0.035388
2440,4,0,1,0.025705
1456,5,0,1,0.018992
1548,3,0,1,0.018903


214 of 4,887 cases exceed the 4/n rule of thumb, but that threshold is a blunt instrument in a
sample this size -- it flags roughly 4% of *any* large dataset almost by construction, not a specific
problem. What matters more is the maximum: 0.044 is nowhere near the "individual point substantially
changes the fitted model" range (values above ~1 are the usual concern). The top cases are all
Richest or Richer women at high burden with a probable-depression outcome -- exactly the profile
driving the interaction finding in Section 8, which makes sense (rare outcome + rare covariate
combination = high leverage by definition) rather than looking like a data problem to fix.

## 22. Sensitivity: extended covariate set (+ Financial Decision-Making + IPV Attitude)

`Financial Decision-Making` and `IPV Attitude` are both in the dataset (see `docs/Dictionary.md`)
but sat outside the primary adjustment set -- the DAG puts `Financial Decision-Making`'s causal role
as ambiguous (plausible confounder of the wealth-depression path, but also plausibly downstream of
wealth itself, which would make adjusting for it a mediator-adjustment bias risk rather than a fix),
and `IPV Attitude` only has DAG edges into Education and Mental Health, not into Wealth, so
conditioning on Education already blocks the one backdoor path it could open.

Rather than leave that as an open question, this section is the direct test: add both to the
adjustment set (13 covariates instead of 11) and see whether anything in Sections 8-13 moves. Neither
addition introduces new complete-separation categories -- checked below before refitting anything.

In [30]:
ext_covariate_cols = covariate_cols + ['Financial Decision-Making', 'IPV Attitude']

for label, df, outcome in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    flags = check_zero_event_cells(df, outcome, ['Financial Decision-Making', 'IPV Attitude'])
    print(f'{label}, new covariates only:', flags if flags else 'none')


Depression, new covariates only: none
Anxiety, new covariates only: none


Clean on both counts, so both go in as ordinary categorical covariates, same reference levels
as everything else (`Financial Decision-Making`: 0 = Husband/other decides; `IPV Attitude`: 0 =
Rejects in all scenarios -- both are the modal, substantively "no autonomy issue / no reported
tolerance" category, consistent with every other reference level in this notebook).

In [31]:
ext_formula = ' + '.join([f'C(Q("{c}"))' for c in ext_covariate_cols])
ext_results = {}

for label, df, ycol in [('Depression', dep, 'Depression_binary'), ('Anxiety', anx, 'Anxiety_binary')]:
    f_m1  = f'{ycol} ~ C(Q("Socioeconomic Status")) + Burden_num + {ext_formula}'
    f_m2  = f'{ycol} ~ C(Q("Socioeconomic Status")) * Burden_num + {ext_formula}'
    f_m1c = f'{ycol} ~ C(Q("Socioeconomic Status")) + Q("Burden_collapsed") + {ext_formula}'
    f_m3  = f'{ycol} ~ C(Q("Socioeconomic Status")) * Q("Burden_collapsed") + {ext_formula}'
    r_m1, _  = fit_svy_logit(f_m1, df)
    r_m2, _  = fit_svy_logit(f_m2, df)
    r_m1c, _ = fit_svy_logit(f_m1c, df)
    r_m3, _  = fit_svy_logit(f_m3, df)
    lr_lin  = lr_test(r_m1, r_m2)
    lr_col  = lr_test(r_m1c, r_m3)
    n_events = int(df[ycol].sum())
    ext_results[label] = dict(r_m1=r_m1, r_m2=r_m2, r_m1c=r_m1c, r_m3=r_m3, lr_lin=lr_lin, lr_col=lr_col,
                               n_events=n_events, epv_m2=n_events / int(r_m2.df_model),
                               converged=all([r_m1.converged, r_m2.converged, r_m1c.converged, r_m3.converged]))
    print(f"{label}: converged={ext_results[label]['converged']} | "
          f"linear-int chi2({lr_lin[1]:.0f})={lr_lin[0]:.2f} p={lr_lin[2]:.4f} | "
          f"collapsed-int chi2({lr_col[1]:.0f})={lr_col[0]:.2f} p={lr_col[2]:.4f} | "
          f"EPV(M2)={ext_results[label]['epv_m2']:.2f}")


Depression: converged=True | linear-int chi2(4)=5.87 p=0.2091 | collapsed-int chi2(8)=18.07 p=0.0207 | EPV(M2)=5.88


Anxiety: converged=True | linear-int chi2(4)=5.66 p=0.2264 | collapsed-int chi2(8)=8.20 p=0.4142 | EPV(M2)=5.53


Everything converges cleanly. Now the interaction terms that actually matter -- specifically
whether Richer x Burden(2+) for depression is still doing what it did in Section 8:

In [32]:
ext_dep_m3_or = or_table(ext_results['Depression']['r_m3'], 'Depression M3 (extended)')
ext_int_rows = ext_dep_m3_or[ext_dep_m3_or['term'].str.contains(':')]
print("Depression Model 3 (extended set): Wealth x Burden_collapsed interaction terms")
display(ext_int_rows.round(3))


Depression Model 3 (extended set): Wealth x Burden_collapsed interaction terms


,model,term,OR,CI_low,CI_high,p
37,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",1.102,0.295,4.117,0.885
38,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",1.194,0.400,3.567,0.750
39,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.708,0.169,2.960,0.636
40,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",2.063,0.578,7.362,0.265
41,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.1]:Q(""Burden_co...",0.599,0.097,3.693,0.581
42,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.2]:Q(""Burden_co...",0.153,0.017,1.376,0.094
43,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.3]:Q(""Burden_co...",0.467,0.093,2.353,0.356
44,Depression M3 (extended),"C(Q(""Socioeconomic Status""))[T.4]:Q(""Burden_co...",0.030,0.003,0.261,0.002


Richer x Burden(2+): OR = 0.030, 95% CI [0.003, 0.261] here, versus OR = 0.028
[0.003, 0.250] in the primary set (Section 8) -- essentially unchanged. `Financial Decision-Making`
and `IPV Attitude` genuinely don't confound this relationship, in either direction, which is the DAG
call in Section 22's opening paragraph turned into an actual number instead of a judgment call left
on the page.

In [33]:
comparison_rows = []
for label, primary_lin, primary_col, primary_epv in [
    ('Depression', (stat, df_diff, p), (stat3, df_diff3, p3), int(res_dep_m1.model.endog.sum()) / int(res_dep_m2.df_model)),
    ('Anxiety', (stat_a, df_diff_a, p_a), (stat_a3, df_diff_a3, p_a3), int(res_anx_m1.model.endog.sum()) / int(res_anx_m2.df_model)),
]:
    ext = ext_results[label]
    comparison_rows.append({
        'Outcome': label,
        'Covariates': '11 (primary)',
        'Linear-int p': round(primary_lin[2], 4),
        'Collapsed-int p': round(primary_col[2], 4),
        'EPV (M2)': round(primary_epv, 2),
    })
    comparison_rows.append({
        'Outcome': label,
        'Covariates': '13 (+FDM +IPV)',
        'Linear-int p': round(ext['lr_lin'][2], 4),
        'Collapsed-int p': round(ext['lr_col'][2], 4),
        'EPV (M2)': round(ext['epv_m2'], 2),
    })
display(pd.DataFrame(comparison_rows))


,Outcome,Covariates,Linear-int p,Collapsed-int p,EPV (M2)
0,Depression,11 (primary),0.1951,0.0180,6.53
1,Depression,13 (+FDM +IPV),0.2091,0.0207,5.88
2,Anxiety,11 (primary),0.2263,0.4036,6.14
3,Anxiety,13 (+FDM +IPV),0.2264,0.4142,5.53


Same qualitative story either way: depression's collapsed-Burden interaction is the one
result that stands out, anxiety shows nothing under either coding, and adding two more covariates
costs about half an event per parameter in EPV without changing any conclusion. `Financial
Decision-Making` and `IPV Attitude` stay out of the primary model -- not because including them
would be wrong, but because the DAG-driven 11-covariate set is the pre-specified analysis and this
section is what confirms that choice isn't hiding anything.

## 23. Exact replication in R / Stata (for final reported standard errors)

Point estimates (odds ratios) from Sections 8-13 match R/Stata exactly, since they don't depend on
the variance estimator -- only the SEs, CIs and p-values do. Those should come from one of the two
below for the submitted manuscript, not from the Python cluster-robust or bootstrap numbers above,
which exist as cross-checks. The companion R notebook (`5. Main Regression Analysis (R).ipynb`) runs
the primary 11-covariate models, the extended 13-covariate sensitivity set from Section 22, and its
own versions of the EPV / multiple-testing / E-value disclosures using R's own design-based p-values
and estimates.

**R (`survey` package):**
```r
library(survey)
dep <- read.csv("resources/depression_dataset.csv")
dep$SES <- relevel(factor(dep$Socioeconomic.Status), ref = "5")

des <- svydesign(
  id = ~PSU, strata = ~Stratum, weights = ~Sampling.weight,
  data = dep, nest = TRUE
)

m1 <- svyglm(Depression_binary ~ SES + Burden_num + Education + Occupation + ...,
             design = des, family = quasibinomial())
m2 <- svyglm(Depression_binary ~ SES * Burden_num + Education + Occupation + ...,
             design = des, family = quasibinomial())
anova(m1, m2, method = "LRT")   # nested-model test, design-based
```

**Stata:**
```stata
import delimited "resources/depression_dataset.csv", clear
svyset psu [pw=sampling_weight], strata(stratum)

svy: logit depression_binary i.ses##c.burden_num i.education i.occupation ...
estat gof                      * design-based Archer-Lemeshow goodness-of-fit (Section 14)
```

`nest = TRUE` (R) / `svyset ..., strata()` (Stata) is exactly the piece `statsmodels` can't do --
that's the whole reason this section exists. `estat gof` is the piece neither Python nor R has a
validated equivalent for (Section 14's version is design-naive) -- Stata is the one place this
notebook defers to another tool rather than reproducing something itself, and that's worth stating
plainly in Methods rather than papering over.

## 24. Interpretation

**Overall interaction test.** The Model 1 vs. Model 2 likelihood-ratio test (Section 13) is the
primary evidence for whether cardiometabolic burden modifies the wealth-mental-health association --
checked before looking at any individual coefficient, because with EPV around 6 (Section 16),
individual interaction-term p-values are individually underpowered and prone to multiple-comparison
noise. On that test, neither outcome shows a significant *linear* interaction. Depression's
*collapsed*-Burden interaction (Model 1c vs. Model 3) does clear p < 0.05 uncorrected (p = 0.018),
but Section 17 treats that honestly as a post-hoc sensitivity finding, not a second pre-registered
primary test -- corrected for multiplicity alongside the other three tests, it lands around p = 0.07.
The right way to report this is as a suggestive finding worth following up, not a confirmed
interaction.

**What the collapsed-Burden result actually looks like.** Section 8's interaction terms and Section 9's
simple slopes tell a specific, not-entirely-linear story: among women with zero or one cardiometabolic
condition, wealthier categories trend toward *somewhat higher* depression odds than the Richest group
(none of these individually significant); once burden reaches two or more conditions, that pattern
reverses hard for the Richer group in particular (OR = 0.028, Section 8). Section 22 confirms this
specific number is essentially unchanged whether `Financial Decision-Making` and `IPV Attitude` are in
the adjustment set or not (0.028 vs. 0.030), so it isn't an artifact of that DAG judgment call either
way. It's also not a large, clean effect the way it might look at first glance -- Section 6 flags real
event sparsity in the driving cell, Section 18's E-value for the confidence-limit version is a much
more modest 7.3 (vs. 68 at the point estimate), and Section 21 confirms the cases carrying the most
leverage are exactly the ones in that rare covariate-outcome combination. All consistent with a real
signal worth reporting, not with a robust, precisely-estimated effect size.

**Anxiety.** No interaction, under any Burden coding, any covariate set, checked here. That's a
negative finding worth reporting as such rather than omitting -- effect modification by
cardiometabolic burden looks like it may be depression-specific rather than a general feature of the
wealth-mental-health relationship in this sample.

**What would change this.** Goodness-of-fit (Section 14) and linearity (Section 15) checks both come
back clean, so a specification error isn't the likely explanation for any of the above. The PHQ-9
>= 10 cutoff (Section 2) is doing real analytical work, though -- Section 19's >= 15 refit is too
sparse (EPV ~ 1.7) to confirm or overturn the finding, so this should be flagged as an open question
in the manuscript rather than resolved one way or the other by that check. Finally, everything above
uses Python's cluster-robust SEs; Section 23's R/Stata replication is what determines whether p = 0.018
survives with a properly design-based (stratified + clustered) standard error, and that number, not
this notebook's, is the one to put in the manuscript.